# 流式调用（stream）

举例


In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


#读取.env配置文件中的信息。
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

#模型的初始化
model = init_chat_model(
    api_base=DEEPSEEK_BASE_URL,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek:deepseek-v4-flash',
)

In [2]:
for chunk in model.stream('帮我解释一下什么是人工智能'):
    print(chunk.text,end='',flush=True) #逐token输出

人工智能（Artificial Intelligence，简称 AI）可以简单理解为：**让机器表现出类似人类智能的行为，去完成通常需要人脑才能完成的任务。**

它不是某一种单一技术，而是一个很大的学科和技术领域，涉及计算机科学、数学、统计学、神经科学、语言学等。核心目标包括让机器能够：

- 感知：看懂图像、听懂语音、识别环境；
- 理解：处理语言、理解文本含义；
- 推理：根据信息做判断；
- 学习：从数据中总结经验；
- 决策：选择行动方案；
- 创造：生成文字、图片、音乐、代码等。

一个常见的理解公式是：

> **AI ≈ 数据 + 算法 + 算力 + 目标**

机器通过大量数据找规律，用算法进行学习，再依靠算力训练模型，最后在具体任务中做预测或决策。

AI 通常可以分为几类：

1. **弱人工智能 / 狭义 AI**  
   只擅长特定任务，比如人脸识别、语音助手、推荐系统、机器翻译、下棋。  
   **目前我们用的绝大多数 AI 都属于这一类。**

2. **强人工智能 / 通用 AI**  
   能像人一样跨领域学习、理解和推理，目前还没有实现。

3. **超人工智能**  
   假想中全面超越人类智能的 AI，目前只存在于讨论和科幻中。

从方法上看，AI 包括：
- **符号主义**：用规则和逻辑推理，比如早期专家系统；
- **连接主义**：用神经网络模拟学习，比如深度学习；
- **行为主义**：通过试错和奖励学习，比如强化学习。

生活中常见的 AI 应用有：手机人脸解锁、语音输入、地图导航、短视频推荐、机器翻译、智能客服、自动驾驶、医疗影像分析，以及像 ChatGPT 这样的生成式 AI。

但要注意：**AI 不等于有意识的人。** 它通常只是在大量数据中寻找统计规律，擅长特定任务，但缺乏真正的理解、常识和因果推理能力，也可能出错、产生偏见或“幻觉”。同时还涉及隐私、安全、伦理和就业等问题。

一句话总结：  
**AI 是用算法和数据让机器模拟智能行为的技术和学科；目前主流 AI 是“狭义 AI”，即在特定任务上很强，但还不是像人一样通用的智能。**

# 2.批量调用（batch）

举例1：一次性接受所有响应

In [5]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch(messages)

for response in responses:
    print(response.content)

你好！我是 **DeepSeek**，一个由深度求索公司开发的 AI 助手。我可以帮你回答问题、写作、翻译、编程、总结信息，或者一起讨论想法。

有什么我可以帮你的吗？
17
中国首都是**北京**。


举例2：按照完成的顺序接收响应

In [7]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch_as_completed(messages)

for response in responses:
    print(response)

(2, AIMessage(content='中国首都是**北京**。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文问题：“中国首都在哪里？”需要直接回答。中国首都是北京。需要简洁。可能注意政策？ 这是事实。可以回答：中国首都是北京。'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 35, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 39, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'c99343d6-ce85-4912-9a28-4d17f9f828b3', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a120de-0e4d-7462-a35a-ff11d1f46172-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 47, 'total_tokens': 82, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'rea

举例3：性能对比

使用batch()

In [11]:
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

# 批量调用（高效）
import time

start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start

print("批量调用结果：")
for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")
print(f"耗时: {batch_time:.2f}秒\n")

批量调用结果：
1. Spring has come.
2. Summer is very hot.
3. Autumn leaves
4. It snows in winter.
耗时: 1.54秒



作为对比，演示循环调用invoke（）

In [12]:
# 循环调用（低效，仅用于对比）
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)
loop_time = time.time() - start

for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")

print(f"循环调用耗时: {loop_time:.2f}秒")

print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")

1. Spring has come.
2. Summer is very hot.
3. Autumn leaves
4. It snows in winter.
循环调用耗时: 4.66秒
批量调用节省: 67.0%
